# OECD parser walkthrough

This notebook is the practical guide for parsing OECD tables in MARIO.

## What this notebook covers

- where each OECD table family comes from;
- when to pass a direct file path and when to pass a directory;
- how `dataset=`, `year=`, and `country=` change the parser workflow;
- the difference between `ICIO`, national `IOT`, and `SUT` parsing;
- the current limitation that OECD tables are parsed as economic tables only.

## Relevant source pages

- OECD ICIO page: [Inter-country input-output tables](https://www.oecd.org/en/data/datasets/inter-country-input-output-tables.html)
- OECD national IOT page: [Input-output tables](https://www.oecd.org/en/data/datasets/input-output-tables.html)
- OECD SUT page: [Supply and use tables](https://www.oecd.org/en/data/datasets/supply-and-use-tables.html)

MARIO does not download OECD files automatically. For `ICIO` and national `IOT`, the expected workflow is to download the source files manually and then point the parser to the local files. For `SUT`, MARIO pulls the data directly from the official OECD SDMX API.

## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_oecd(...)`

The same function currently supports:

- OECD `ICIO` csv bundles;
- OECD national total-table `IOT` csv files;
- OECD `SUT` tables through the official SDMX API.

## Key arguments

The key public arguments are:

- `path`: one local OECD file or one directory containing multiple OECD files;
- `dataset`: choose `"ICIO"`, `"IOT"`, or `"SUT"`;
- `year`: used to disambiguate directories and required for `dataset="SUT"`;
- `country`: used to disambiguate national IOT files and required for `dataset="SUT"`.

## Direct file path vs directory path

Use a **direct file path** when you already know the exact OECD file to parse. This is the simplest workflow and usually means you do not need any further disambiguation.

Use a **directory path** when you keep several OECD files together. In that case MARIO scans the directory and then uses `dataset=`, `year=`, and sometimes `country=` to pick the correct candidate.

## Expected local layouts and caveats

Typical local layouts are:

For `dataset="ICIO"`:

```text
OECD/ICIO/
├── Extended/
│   └── 2022.csv
└── Regular/
    └── 2022_SML.csv
```

For `dataset="IOT"`:

```text
OECD/IOT/TTL/
└── ITA2022ttl.csv
```

For `dataset="SUT"`, `path` is not required; use `country=` and `year=` to select the SDMX slice.

Practical caveats:

- the `IOT` workflow supports only the OECD accounts labeled as total table;
- in the extended `ICIO` release, `CN1` and `CN2` are aggregated automatically into `CHN`, and `MX1` and `MX2` into `MEX`.

In [1]:
import mario

## Parse one explicit OECD ICIO csv

Use this when you want to point MARIO to one exact `ICIO` file, either regular or extended.

In [2]:
db = mario.parse_oecd(
    path="/path/to/2022_SML.csv",
    dataset="ICIO",
)

db

INFO Parser: reading OECD ICIO file 2022_SML.csv.


INFO Parser: OECD ICIO parsed with 81 sector regions, 50 sectors, 486 final-demand columns and 2 factor rows.


INFO Metadata: initialized.


name = OECD ICIO 2022 regular
table = IOT
scenarios = ['baseline']
Factor of production = 2
Satellite account = 1
Consumption category = 6
Region = 81
Sector = 50

## Parse one national OECD total-table IOT

Use this when you want one national total table such as `CZE2014ttl.csv`.

In [3]:
db = mario.parse_oecd(
    path="/path/to/ITA2022ttl.csv",
    dataset="IOT",
)

db

INFO Parser: reading OECD IOT file ITA2022ttl.csv.


INFO Parser: OECD national IOT parsed with 50 sectors, 9 final-demand columns and 6 factor rows.


INFO Metadata: initialized.


name = OECD IOT ITA 2022
table = IOT
scenarios = ['baseline']
Factor of production = 6
Satellite account = 1
Consumption category = 9
Region = 1
Sector = 50

## Parse one OECD SUT directly from the SDMX API

In principle, this workflow does not require a local `path`, but it does require `country=` and `year=`.

At the moment we do **not** show an executable example in the documentation because the official OECD SDMX endpoint can intermittently refuse the request with HTTP errors such as `403 Forbidden`. When that happens, the issue is upstream of MARIO's SUT transformation logic and the parse cannot complete.

For this reason, treat `dataset="SUT"` as temporarily unstable in the public docs until the OECD SDMX service becomes reliably accessible again.

In [4]:
db = mario.parse_oecd(
    dataset="SUT",
    country="ITA",
    year=2022,
)

db

INFO Parser: requesting OECD SDMX OECD.SDD.NAD,DSD_NASU@DF_SUPPLY_T1500,2.0 for ITA 2022.


INFO Parser: requesting OECD SDMX OECD.SDD.NAD,DSD_NASU@DF_USEPP_T1600,2.0 for ITA 2022.


INFO Parser: requesting OECD SDMX OECD.SDD.NAD,DSD_NASU@DF_USEVA_T1600,2.0 for ITA 2022.


INFO Parser: OECD SUT payload ready with shapes S=(65, 65), U=(65, 65), Yc=(65, 7), Va=(9, 65), Vc=(9, 65); commodity_diff=0.6000000000058208, activity_diff=151254.30000000002.


INFO Metadata: initialized.


name = OECD SUT ITA 2022
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 65
Commodity = 65
Factor of production = 9
Satellite account = 1
Consumption category = 7
Region = 1

## Environmental extensions

**Warning.** The OECD parser currently exposes the economic tables only.

`ICIO`, national `IOT`, and `SUT` tables are therefore **not** parsed as environmentally extended tables in MARIO at the moment.